MAT550 - Final Project - Grover

# US Labor Market System

## instructions

Conduct a complete applied time series study in a real operational context where at least three related series are monitored. Treat the series both as individual processes and as components of a system, producing forecasts, diagnostics, and interpretable evidence in a professional or research setting. The project should result in two deliverables, the code with models and interpretations, and an interactive dashboard implemented in Streamlit
that allows a practitioner to explore the data, reproduce forecasts for a selected horizon,and inspect model diagnostics. The narrative of the project should move from the business or scientific question, through data preparation and exploratory analysis, into model identification, estimation, validation, and finally forecast communication.

The chosen context must be documented with enough detail for a reader outside the team to understand why the forecasting exercise matters and what decisions the forecasts are intended to support.

The analysis must compare, at a minimum, three families of models covered during the semester. The first is exponential smoothing, including simple, Holt, and Holt-Winters variants. The second is the Box-Jenkins methodology, comprising AR, MA, ARMA, and ARIMA. The third is the machine learning family, which includes tree-based models and one neural network.

The Streamlit application should expose, at a minimum, a selector for the target series, a control for the forecast horizon, a visualization of the historical data with overlaid forecasts and prediction intervals, a panel with residual diagnostics, and a summary table comparing accuracy across the implemented models. The dashboard is evaluated on clarity and on its fitness as a communication tool for a non-technical
decision maker, not on visual ornamentation.

The potential data sources include energy and utilities, macroeconomics and finance, public health and epidemiology, transportation and mobility, retail and demand forecasting, environment and climate, and tourism and hospitality. Students may propose alternative sources provided that the data are publicly accessible.

Students submit a public repository containing a reproducibility notebook and a link to the deployed Streamlit dashboard. Other libraries and deployment platforms are accepted,such as Shiny (R), Google Cloud, AWS, and Azure. A 5 minutes max video is required to present the results. In the video, do not explain the code; focus on the project's motivation, results, and impact in the selected context.

## context

The US labor market operates as an interconnected system where three key metrics, Job Openings, Hires, and Quits, are often treated as important indicators of the broader economy. Job Openings represent corporate demand and business growth, signaling a company's need to expand its workforce. Hires indicate fulfilled demand, tracking the actual flow of workers into new roles and the market's ability to successfully match talent to opportunity. Finally, Quits serve as a barometer for worker confidence; a high quit rate implies employees feel secure enough to leave their current roles for better compensation or conditions, which in turn forces businesses to post new openings to replace them. Together, these three series form a continuous loop of labor supply and demand, making it essential to analyze them as an integrated system rather than in isolation.

## necessary libraries

In [ ]:
pip install pmdarima

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pmdarima as pm
import statsmodels.api as sm
import scipy.stats as stats
import joblib
import os

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from statsmodels.stats.diagnostic import acorr_ljungbox

# **data**

from JOLTS (https://www.bls.gov/jlt/home.htm)

In [ ]:
# load data
openings = pd.read_csv('job_openings.csv', index_col = 'date')
hires = pd.read_csv('hires.csv', index_col = 'date')
quits = pd.read_csv('quits.csv', index_col = 'date')


# convert index to datetime and set frequency
for df in [openings, hires, quits]:
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)

In [ ]:
# merging into a single 'system' dataframe for easier multivariate analysis
df = pd.concat([openings, hires, quits], axis=1)
df.columns = ['Openings', 'Hires', 'Quits']
df = df.asfreq('MS')

# verify
print(df.head(3))
print("\n", openings.head(3))
print("\n", hires.head(3))
print("\n", quits.head(3))

## eda

In [ ]:
print(df.info(), '\n=============================================================\n')
print(hires.info(), '\n=============================================================\n')
print(quits.info(), '\n=============================================================\n')
print(openings.info())

In [ ]:
print(df.describe(),  '\n=============================================================\n')
print(hires.describe(),  '\n=============================================================\n')
print(quits.describe(),  '\n=============================================================\n')
print(openings.describe())

In [ ]:
# individual plots
plt.figure(figsize=(15, 6))
plt.subplot(1, 3, 1)

plt.plot(openings)
plt.title('Monthly Job Openings (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate')
plt.xticks([])

plt.subplot(1, 3, 2)

plt.plot(hires)
plt.title('Monthly Hires (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate')
plt.xticks([])

plt.subplot(1, 3, 3)

plt.plot(quits)
plt.title('Monthly Quits (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate')
plt.xticks([])

plt.tight_layout()
plt.show()

In [ ]:
# system plot
plt.figure(figsize=(12, 6))

plt.plot(df['Openings'], label='Openings', color='blue')
plt.plot(df['Hires'], label='Hires', color='yellow')
plt.plot(df['Quits'], label='Quits', color='red')

plt.title('Monthly Job Openings, Hires, and Quits (2000-2026)')
plt.xlabel('Date')
plt.ylabel('Rate (%)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap
plt.figure(figsize=(8, 6))

sns.heatmap(df.corr(), annot=True, cmap='RdBu_r', center=0)

plt.title("Systemic Correlation: Openings, Hires, and Quits")

plt.tight_layout()
plt.show()

In [ ]:
# decomposing Hires (likely has strong seasonality)
decomp = seasonal_decompose(df['Hires'], model='additive')

fig = decomp.plot()
fig.set_size_inches(12, 8)

plt.show()

In [ ]:
def check_stationarity(series, name):
    res = adfuller(series.dropna())
    print(f"--- {name} ---")
    print(f"ADF Statistic: {res[0]:.4f}")
    print(f"p-value: {res[1]:.4f}")
    if res[1] <= 0.05:
        print("Result: Stationary")
    else:
        print("Result: Non-Stationary (Needs Differencing)")

for col in df.columns:
    check_stationarity(df[col], col)

# **models**

## exponential smoothing

In [ ]:
def train_hw_model(train_data, test_data, horizon=24):
    """
    Trains a Holt-Winters model and returns forecast and metrics.
    """
    # 1. Fit the model (Using Additive Trend and Seasonality for Rates)
    model = ExponentialSmoothing(
        train_data,
        trend='add',
        seasonal='add',
        seasonal_periods=12
    ).fit(optimized=True)

    # 2. Generate Forecast
    forecast = model.forecast(horizon)

    # 3. Calculate Metrics
    mae = mean_absolute_error(test_data, forecast)
    rmse = np.sqrt(mean_squared_error(test_data, forecast))

    return model, forecast, mae, rmse

# --- Application to all three series ---
hw_results = {}

for col in df.columns:
    train = df[col].iloc[:-24]
    test = df[col].iloc[-24:]

    model, forecast, mae, rmse = train_hw_model(train, test)

    # Store results in a dictionary for easy access by the dashboard
    hw_results[col] = {
        'model': model,
        'forecast': forecast,
        'mae': mae,
        'rmse': rmse,
        'test_actual': test
    }

print("Holt-Winters models trained for all three series.")

In [ ]:
def plot_hw_diagnostics(series_name):
    res_entry = hw_results[series_name]
    residuals = res_entry['test_actual'] - res_entry['forecast']

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    # Residual Plot
    ax[0].plot(residuals)
    ax[0].axhline(0, color='black', linestyle='--')
    ax[0].set_title(f'Residuals: {series_name}')

    # Histogram (Checking for Normal Distribution)
    ax[1].hist(residuals, bins=15, edgecolor='black')
    ax[1].set_title('Error Distribution')

    plt.tight_layout()
    plt.show()

# Test it
plot_hw_diagnostics('Openings')

## Box-Jenkins

In [ ]:
# create a folder to store the saved models
if not os.path.exists('models'):
    os.makedirs('models')

def train_arima_model(train_data, test_data, horizon=24):
    """
    Automates the Box-Jenkins identification and estimation process.
    """
    # auto_arima handles the 'Identification' (differencing and lags)
    # and 'Estimation' (AIC optimization) steps.
    model = pm.auto_arima(
        train_data,
        seasonal=True, m=12,  # Monthly seasonality
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        trace=False
    )

    # Generate forecast and confidence intervals
    forecast, conf_int = model.predict(n_periods=horizon, return_conf_int=True)

    # Calculate Metrics
    mae = mean_absolute_error(test_data, forecast)
    rmse = np.sqrt(mean_squared_error(test_data, forecast))

    return model, forecast, conf_int, mae, rmse

# Store ARIMA results
arima_results = {}

print("Starting ARIMA training and saving process. This may take a few minutes...")

for col in df.columns:
    print(f"Training ARIMA for {col}...")
    train = df[col].iloc[:-24]
    test = df[col].iloc[-24:]

    model, forecast, conf, mae, rmse = train_arima_model(train, test)

    # Save the trained model to a file
    filename = f'models/arima_{col}.pkl'
    joblib.dump(model, filename, compress=9)
    print(f"Saved: {filename}")

    arima_results[col] = {
        'model': model,
        'forecast': forecast,
        'conf_int': conf,
        'mae': mae,
        'rmse': rmse,
        'summary': model.summary()
    }

print("\nBox-Jenkins (ARIMA) models estimated and saved for all three series.")

In [ ]:
def plot_arima_diagnostics(series_name):
    model = arima_results[series_name]['model']

    # This built-in function satisfies the "residual diagnostics" requirement perfectly
    # It shows: Standardized residuals, Histogram, Q-Q plot, and Correlogram
    model.plot_diagnostics(figsize=(12, 8))
    plt.suptitle(f"ARIMA Diagnostic Panel: {series_name}")
    plt.tight_layout()
    plt.show()

# Example check
plot_arima_diagnostics('Hires')

## machine learning

In [ ]:
def create_multivariate_lags(df, target_column, n_lags=3):
    """
    Creates a feature set using lags from ALL three series
    to predict a single target.
    """
    X, y = [], []
    data = df.values
    target_idx = df.columns.get_loc(target_column)

    for i in range(n_lags, len(data)):
        # Feature vector: all variables at t-1, t-2, ... t-n
        X.append(data[i-n_lags:i].flatten())
        y.append(data[i, target_idx])

    return np.array(X), np.array(y)

# Prepare data for 'Hires'
X, y = create_multivariate_lags(df, target_column='Hires', n_lags=6)

# Split (Same as before, keep the last 24 months for testing)
X_train, X_test = X[:-24], X[-24:]
y_train, y_test = y[:-24], y[-24:]

In [ ]:
print("Training and saving XGBoost models...")

# We use n_lags=6 because that's what you established in your notebook!
n_lags = 6

for col in df.columns:
    # 1. Create the multivariate feature matrix
    X, y = create_multivariate_lags(df, target_column=col, n_lags=n_lags)

    # 2. Train on all data EXCEPT the last 24 months (our test set)
    X_train = X[:-24]
    y_train = y[:-24]

    # 3. Fit the model
    model_xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5)
    model_xgb.fit(X_train, y_train)

    # 4. Save with maximum compression
    filename = f'models/xgb_{col}.pkl'
    joblib.dump(model_xgb, filename, compress=9)
    print(f"Saved: {filename}")

print("All Machine Learning models are ready for deployment!")

In [ ]:
# Initialize dictionary to store XGBoost forecasts and test actuals
xgb_forecasts = {}

print("Generating XGBoost forecasts...")

for col in df.columns:
    # Load the trained XGBoost model
    filename = f'models/xgb_{col}.pkl'
    loaded_xgb_model = joblib.load(filename)

    # Prepare the test data (X_test) for the current column
    # We need to recreate X_test for each target column
    X_full, y_full = create_multivariate_lags(df, target_column=col, n_lags=n_lags)
    X_test_current = X_full[-24:] # Last 24 months for testing
    y_test_current = y_full[-24:] # Last 24 months for actuals

    # Make predictions on the X_test_current data
    xgb_pred = loaded_xgb_model.predict(X_test_current)

    # Store the forecast and actuals
    xgb_forecasts[col] = {
        'forecast': xgb_pred,
        'test_actual': y_test_current
    }

print("XGBoost forecasts generated.")

In [ ]:
results = pd.DataFrame({
    'Actual': y_test,
    'Holt-Winters': hw_results['Hires']['forecast'].values,
    'ARIMA': arima_results['Hires']['forecast'],
    'XGBoost': xgb_forecasts['Hires']['forecast'],
}, index=test.index)

# Calculate RMSE for all
final_metrics = results.apply(lambda x: np.sqrt(mean_squared_error(y_test, x))).drop('Actual')
print("Final RMSE Leaderboard:")
print(final_metrics.sort_values())

In [ ]:
def run_ml_diagnostics(actual, predicted, model_name):
    residuals = actual - predicted

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2)

    # 1. Residuals Over Time
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(residuals)
    ax1.axhline(0, color='red', linestyle='--')
    ax1.set_title(f'{model_name}: Residuals Over Time')

    # 2. Histogram / Normality
    ax2 = fig.add_subplot(gs[0, 1])
    sns.histplot(residuals, kde=True, ax=ax2)
    ax2.set_title('Distribution of Errors')

    # 3. Autocorrelation (ACF) - CRITICAL for Time Series
    ax3 = fig.add_subplot(gs[1, 0])
    sm.graphics.tsa.plot_acf(residuals, lags=20, ax=ax3)
    ax3.set_title('Residual Autocorrelation (ACF)')

    # 4. Q-Q Plot (Are residuals normally distributed?)
    ax4 = fig.add_subplot(gs[1, 1])
    stats.probplot(residuals, dist="norm", plot=ax4)
    ax4.set_title('Normal Q-Q Plot')

    plt.tight_layout()
    plt.show()

# How to use it:
run_ml_diagnostics(xgb_forecasts['Hires']['test_actual'], xgb_forecasts['Hires']['forecast'], "XGBoost")

In [ ]:
def formal_test(actual, predicted):
    residuals = actual - predicted
    # We test if the first 10 lags are independent
    lb_test = acorr_ljungbox(residuals, lags=[10])
    p_value = lb_test.lb_pvalue.values[0]

    if p_value > 0.05:
        return f"Pass (p={p_value:.3f}): Residuals are White Noise."
    else:
        return f"Fail (p={p_value:.3f}): Residuals have remaining structure."

print(f"XGBoost Test: {formal_test(xgb_forecasts['Hires']['test_actual'], xgb_forecasts['Hires']['forecast'])}")